# Create Feature Importance for Robustness Dropout

In [18]:
import anndata as ad
import scanpy as sc
from sklearn.linear_model import LogisticRegression
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)


# Filtering classes with >=150 samples
#y = adata.obs['scumi-annotation']

#min_samples = 150
#class_counts = y.value_counts()
#keep_classes = class_counts[class_counts >= min_samples].index

#adata = adata[adata.obs["scumi-annotation"].isin(keep_classes)].copy()


# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.X#to_df()
gene_names_train = adata_train.var_names
#y_train = adata_train.obs['Manually_curated_celltype']
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
#y_test = adata_test.obs['Manually_curated_celltype']
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
Before filtering highly variable genes 

In [1]:
import json

with open('../scumi-dev/R/marker_gene/human_pbmc_marker.json', 'r') as file:
    marker_genes = json.load(file)

print(marker_genes)

{'CD4+ T cell': ['CD3D+', 'CD3E+', 'CD3G+', 'TRAC+', 'CD4+', 'TCF7+', 'CD27+', 'IL7R+', 'CD8A-', 'CD8B-', 'GNLY-', 'NKG7-', 'CST7-'], 'Cytotoxic T cell': ['CD3D+', 'CD3E+', 'CD3G+', 'TRAC+', 'CD8A+', 'CD8B+', 'GZMK+', 'CCL5+', 'NKG7+', 'CD4-', 'FCER1G-'], 'B cell': ['CD19+', 'MS4A1+', 'CD79A+', 'CD79B+', 'MZB1+', 'IGHD+', 'IGHM+'], 'Natural killer cell': ['NCAM1+', 'NKG7+', 'KLRB1+', 'KLRD1+', 'KLRF1+', 'KLRC1+', 'KLRC2+', 'KLRC3+', 'KLRC4+', 'CD3D-', 'CD3E-', 'CD3G-', 'CD14-', 'FCGR3A+', 'FCGR3B+', 'ITGAL+', 'ITGAM+', 'FCER1G+', 'TRAC-'], 'CD14+ monocyte': ['VCAN+', 'FCN1+', 'S100A8+', 'S100A9+', 'CD14+', 'ITGAL+', 'ITGAM+', 'CSF3R+', 'CSF1R+', 'CX3CR1+', 'FCGR3A-', 'FCGR3B-', 'TYROBP+', 'LYZ+', 'S100A12+', 'CD3D-', 'CD3E-', 'CD3G-', 'TRAC-', 'NKG7-', 'KLRB1-', 'KLRD1-'], 'CD16+ monocyte': ['FCN1+', 'FCGR3A+', 'FCGR3B+', 'ITGAL+', 'ITGAM+', 'CSF3R+', 'CSF1R+', 'CX3CR1+', 'CDKN1C+', 'MS4A7+', 'S100A8-', 'S100A9-', 'S100A12-', 'CD14-', 'CD3D-', 'CD3E-', 'CD3G-', 'TRAC-', 'NKG7-', 'KLRB1

## Option 1: get random order of marker genes through a set

In [19]:
marker_genes_list = []
for cell_class in marker_genes.values():
    for marker_gene in cell_class:
        marker_genes_list.append(marker_gene[:-1])

marker_genes_set = set(marker_genes_list)

marker_genes_set_filtered = [g for g in marker_genes_set if g in adata_train.var_names]

print(marker_genes_set)
print(len(marker_genes_set))

{'TCF4', 'KLRB1', 'ITGA2B', 'CLEC10A', 'CD14', 'CLEC4C', 'ITGAM', 'CD1C', 'GNLY', 'MS4A1', 'KLRC3', 'NRGN', 'CD38', 'IGHA1', 'TYROBP', 'FCGR3A', 'IL7R', 'HLA-DPB1', 'GZMB', 'TCF7', 'CD79A', 'MYL9', 'S100A12', 'CDKN1C', 'KLRC2', 'CD3E', 'TRAC', 'CD4', 'PF4', 'NKG7', 'SLAMF7', 'GZMK', 'FCN1', 'CD8A', 'FCGR3B', 'MS4A7', 'CD8B', 'HLA-DPA1', 'LILRA4', 'CD3G', 'HLA-DQA1', 'KLRC1', 'VCAN', 'IGHM', 'RGS18', 'XBP1', 'IGHG1', 'FCER1A', 'CD3D', 'SPARC', 'GP5', 'KLRC4', 'CD19', 'JCHAIN', 'CD79B', 'IL3RA', 'CX3CR1', 'CSF1R', 'MZB1', 'IGHD', 'KLRF1', 'IGHA2', 'LYZ', 'ITGAL', 'FCER1G', 'CST7', 'NCAM1', 'S100A8', 'IGHG3', 'PPBP', 'CD1E', 'FCGR2B', 'ITGAX', 'GNG11', 'KLRD1', 'TUBB1', 'S100A9', 'CSF3R', 'IGHG2', 'CD27', 'IRF7', 'IGHG4', 'CCL5'}
83


In [31]:
import scanpy as sc

# Compute differential gene expression
sc.tl.rank_genes_groups(adata_train, groupby='scumi-annotation', method='wilcoxon')


de_names_df = pd.DataFrame(adata_train.uns['rank_genes_groups']['names'])

de_genes_ordered = []
max_rows = de_names_df.shape[0]  # Number of genes per cell type
cell_types = de_names_df.columns  # List of all cell types

# Iterate over the rows and thereby interleave the cell types
for row_idx in range(max_rows):
    for c_type in cell_types:
        gene = de_names_df.iloc[row_idx][c_type]
        if gene not in de_genes_ordered:
            de_genes_ordered.append(gene)

print(de_genes_ordered[:5])

['CD74', 'EEF1A1', 'LYZ', 'GZMA', 'HLA-DRA']


In [32]:
master_feature_importance = list(marker_genes_set_filtered)

for gene in de_genes_ordered:
    if gene not in master_feature_importance:
        master_feature_importance.append(gene)

print(f"Number of unique marker genes: {len(marker_genes_set_filtered)}")
print(f"Number of marker genes in Feature Importance:     {len(master_feature_importance)}")

Number of unique marker genes: 82
Number of marker genes in Feature Importance:     10000


In [26]:
import pickle

output_filename = "master_feature_importance.pkl"
with open(output_filename, "wb") as f:
    pickle.dump(master_feature_importance, f)

## Option 2: interleave marker genes

In [33]:
marker_genes_interleaved = []

# Compute maximum number of genes per cell type
max_marker_rows = max(len(genes) for genes in marker_genes.values())
cell_classes = list(marker_genes.keys())

# interleave the marker genes from the cell types
for row_idx in range(max_marker_rows):
    for cell_class in cell_classes:
        # Check if current cell type still has marker genes left
        if row_idx < len(marker_genes[cell_class]):
            raw_gene = marker_genes[cell_class][row_idx]
            clean_gene = raw_gene[:-1]
            
            if clean_gene in adata_train.var_names and clean_gene not in marker_genes_interleaved:
                marker_genes_interleaved.append(clean_gene)

In [34]:
import scanpy as sc

# Compute differential gene expression
sc.tl.rank_genes_groups(adata_train, groupby='scumi-annotation', method='wilcoxon')


de_names_df = pd.DataFrame(adata_train.uns['rank_genes_groups']['names'])

de_genes_ordered = []
max_rows = de_names_df.shape[0]  # Number of genes per cell type
cell_types = de_names_df.columns  # List of all cell types

# Iterate over the rows and thereby interleave the cell types
for row_idx in range(max_rows):
    for c_type in cell_types:
        gene = de_names_df.iloc[row_idx][c_type]
        if gene not in de_genes_ordered:
            de_genes_ordered.append(gene)

print(de_genes_ordered[:5])

['CD74', 'EEF1A1', 'LYZ', 'GZMA', 'HLA-DRA']


In [35]:
master_feature_importance = marker_genes_interleaved

for gene in de_genes_ordered:
    if gene not in master_feature_importance:
        master_feature_importance.append(gene)

print(f"Number of unique marker genes: {len(marker_genes_set_filtered)}")
print(f"Number of marker genes in Feature Importance:     {len(master_feature_importance)}")

Number of unique marker genes: 82
Number of marker genes in Feature Importance:     10000


In [30]:
import pickle

output_filename = "master_feature_importance_interleaved_marker_genes.pkl"
with open(output_filename, "wb") as f:
    pickle.dump(master_feature_importance, f)